# Disease Symptom Classifier — EDA & Training Walkthrough

**Project:** Medical Symptom Classifier using Naive Bayes from Scratch  
**Author:** Sajid Ansari  
**Stack:** Python · NumPy · Pandas · Plotly · Streamlit

---

## Objective
Build a multi-class Naive Bayes classifier that predicts a disease from binary symptom inputs.  
All core ML code (NB classifiers, metrics, preprocessing) is written from scratch — no scikit-learn for the algorithm.

## Table of Contents
1. Data Loading & Overview  
2. Exploratory Data Analysis  
3. Preprocessing  
4. Naive Bayes Theory  
5. Model Training — All Three Variants  
6. Evaluation & Comparison  
7. The Zero-Probability Problem & Laplace Smoothing  
8. Log-Space Computation  
9. Top-K Predictions  
10. Saving the Model

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys, os
sys.path.insert(0, '..')

from my_library import (
    BernoulliNB, MultinomialNB, GaussianNB,
    load_and_prepare, train_test_split_stratified, encode_labels,
    accuracy, confusion_matrix, classification_report, precision_recall_f1,
)

print('All imports successful!')

## 1. Data Loading & Overview

In [ ]:
df = pd.read_csv('../data/disease_symptom_dataset.csv')
symptom_cols = [c for c in df.columns if c != 'prognosis']

print(f'Shape: {df.shape}')
print(f'Diseases: {df["prognosis"].nunique()}')
print(f'Symptoms: {len(symptom_cols)}')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Class distribution
counts = df['prognosis'].value_counts().reset_index()
counts.columns = ['Disease', 'Count']

fig = px.bar(
    counts, x='Count', y='Disease', orientation='h',
    title='Samples per Disease (balanced dataset)',
    color='Count', color_continuous_scale='Blues',
    height=900
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Top 20 most common symptoms across all records
symptom_freq = df[symptom_cols].sum().sort_values(ascending=False).head(20)

fig = px.bar(
    x=symptom_freq.values,
    y=[s.replace('_', ' ').title() for s in symptom_freq.index],
    orientation='h',
    title='Top 20 Most Frequent Symptoms',
    labels={'x': 'Frequency', 'y': 'Symptom'},
    color=symptom_freq.values,
    color_continuous_scale='Teal',
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [ ]:
# Average number of symptoms per disease
avg_symptoms = df.groupby('prognosis')[symptom_cols].mean().sum(axis=1).sort_values(ascending=False)

fig = px.bar(
    x=avg_symptoms.values,
    y=avg_symptoms.index,
    orientation='h',
    title='Average Number of Symptoms per Disease',
    labels={'x': 'Avg Symptoms', 'y': 'Disease'},
    height=700
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 3. Preprocessing

In [ ]:
X, y, symptom_cols, classes = load_and_prepare('../data/disease_symptom_dataset.csv')

X_train, X_test, y_train, y_test = train_test_split_stratified(
    X, y, test_size=0.20, random_state=42
)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')

# Verify stratification
train_counts = pd.Series(y_train).value_counts()
test_counts  = pd.Series(y_test).value_counts()
print(f'Train samples per class — min: {train_counts.min()}, max: {train_counts.max()}')
print(f'Test  samples per class — min: {test_counts.min()},  max: {test_counts.max()}')

## 4. Naive Bayes Theory

### Bayes' Theorem
$$P(disease \mid \mathbf{x}) = \frac{P(\mathbf{x} \mid disease) \cdot P(disease)}{P(\mathbf{x})}$$

The denominator $P(\mathbf{x})$ is constant across all classes, so:
$$P(disease \mid \mathbf{x}) \propto P(disease) \cdot P(\mathbf{x} \mid disease)$$

### Naive Independence Assumption
$$P(\mathbf{x} \mid disease) = \prod_{j=1}^{d} P(x_j \mid disease)$$

### Bernoulli NB (our best model)
For binary features where $x_j \in \{0, 1\}$:
$$P(x_j \mid c) = p_{jc}^{x_j} \cdot (1 - p_{jc})^{1 - x_j}$$

where $p_{jc} = P(x_j = 1 \mid c)$ estimated from training data with Laplace smoothing:
$$\hat{p}_{jc} = \frac{N_{jc} + \alpha}{N_c + 2\alpha}$$

## 5. Model Training — All Three Variants

In [ ]:
# Train all variants
models = {
    'BernoulliNB (α=1.0)':  BernoulliNB(alpha=1.0),
    'BernoulliNB (α=0.5)':  BernoulliNB(alpha=0.5),
    'MultinomialNB (α=1.0)': MultinomialNB(alpha=1.0),
    'GaussianNB':            GaussianNB(),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc_val = accuracy(y_test, y_pred)
    m = precision_recall_f1(y_test, y_pred, classes, average='macro')
    results[name] = {'model': model, 'acc': acc_val, 'f1': m['f1'], 'y_pred': y_pred}
    print(f'{name:<28}  Accuracy: {acc_val:.4f}  Macro-F1: {m["f1"]:.4f}')

## 6. Evaluation & Comparison

In [ ]:
# Comparison chart
model_names = list(results.keys())
accs = [results[n]['acc'] * 100 for n in model_names]
f1s  = [results[n]['f1']  * 100 for n in model_names]

fig = go.Figure([
    go.Bar(name='Accuracy', x=model_names, y=accs, marker_color='#3b82f6'),
    go.Bar(name='Macro F1', x=model_names, y=f1s,  marker_color='#10b981'),
])
fig.update_layout(
    barmode='group', title='Model Comparison',
    yaxis=dict(title='Score (%)', range=[70, 101]),
    legend=dict(x=0.75, y=1)
)
fig.show()

In [ ]:
# Full classification report for best model
best_model = results['BernoulliNB (α=1.0)']['model']
y_pred_best = results['BernoulliNB (α=1.0)']['y_pred']
print(classification_report(y_test, y_pred_best, classes))

In [ ]:
# Confusion matrix heatmap (subset — top 15 diseases for readability)
cm = confusion_matrix(y_test, y_pred_best, classes)

fig = px.imshow(
    cm,
    labels=dict(x='Predicted', y='Actual', color='Count'),
    x=classes.tolist(),
    y=classes.tolist(),
    color_continuous_scale='Blues',
    title='Confusion Matrix — BernoulliNB',
    height=900,
)
fig.update_xaxes(tickangle=45)
fig.show()

## 7. The Zero-Probability Problem & Laplace Smoothing

Without smoothing, if a symptom never co-occurs with a disease in training data,  
$P(symptom \mid disease) = 0$, making the entire posterior 0 regardless of other evidence.

**Laplace smoothing** adds a pseudo-count $\alpha$ to every feature:
$$\hat{p}_{jc} = \frac{N_{jc} + \alpha}{N_c + \alpha \cdot V}$$

where $V$ is the vocabulary size (number of features).

In [ ]:
# Demo: what happens without smoothing
print('Without smoothing (alpha=0):')
nb_no_smooth = BernoulliNB(alpha=0.0)
nb_no_smooth.fit(X_train, y_train)
y_demo = nb_no_smooth.predict(X_test[:5])
print('Predictions (biased by -inf):', y_demo)
print('Many predictions collapse to one class due to -inf log-probs')

print('\nWith smoothing (alpha=1.0):')
nb_smooth = BernoulliNB(alpha=1.0)
nb_smooth.fit(X_train, y_train)
y_smooth = nb_smooth.predict(X_test[:5])
print('Predictions:', y_smooth)
print('True labels:', y_test[:5])

## 8. Log-Space Computation

With $d = 131$ features, naively computing $\prod_{j=1}^{131} p_j$ can underflow float64:

If each $p_j \approx 0.1$, then $\prod p_j \approx 10^{-131}$ — below `float64` min (~$10^{-308}$, but precision is lost much earlier).

**Solution:** Work in log-space, convert multiplication to addition:
$$\log P(c \mid \mathbf{x}) \propto \log P(c) + \sum_j \log P(x_j \mid c)$$

In [ ]:
# Demonstrate underflow
small_probs = [0.05] * 131
naive_product = 1.0
for p in small_probs:
    naive_product *= p

log_sum = sum(np.log(p) for p in small_probs)

print(f'Naive product:    {naive_product}   (underflows to 0!)')
print(f'Log-space sum:    {log_sum:.4f}')
print(f'Recovered value:  {np.exp(log_sum):.2e}   (still tiny, but we only need the argmax)')

## 9. Top-K Predictions

In [ ]:
# Show top-3 predictions for 5 test samples
top3_all = best_model.top_k_predictions(X_test[:5], k=3)

for i, preds in enumerate(top3_all):
    print(f'\nSample {i+1} — True label: {y_test[i]}')
    for rank, (disease, prob) in enumerate(preds, 1):
        marker = ' ← CORRECT' if disease == y_test[i] else ''
        print(f'  #{rank}: {disease:<45} {prob:.4f}{marker}')

## 10. Saving the Model

In [ ]:
import pickle

payload = {
    'model':        best_model,
    'model_name':   'BernoulliNB (α=1.0)',
    'symptom_cols': list(symptom_cols),
    'classes':      classes,
    'accuracy':     results['BernoulliNB (α=1.0)']['acc'],
    'macro_f1':     results['BernoulliNB (α=1.0)']['f1'],
}

with open('../model.pkl', 'wb') as f:
    pickle.dump(payload, f)

print('Model saved to model.pkl')
print(f'Accuracy: {payload["accuracy"]:.4f}')
print(f'Macro F1: {payload["macro_f1"]:.4f}')